# Correction — Phase 1 : nouveau score d'anomalie (sans réentraînement)

Suite au diagnostic du notebook 07 : 4 catégories de défauts (`contamination`,
`crack`, `faulty_imprint`, `scratch`) ont un rappel de 0 % avec le score actuel
(moyenne globale de l'erreur de reconstruction). Ce notebook implémente et évalue la
correction validée empiriquement : un score basé sur un **centile élevé de l'erreur,
après soustraction d'une référence saine** (`healthy_baseline` /
`image_scores_pooled`, ajoutées à `anomaly.py`).

**Aucun réentraînement** : le modèle auto-encodeur existant (notebook 04) est
rechargé tel quel ; seule la façon de calculer le score d'anomalie change.

In [1]:
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

from indusense.vision.anomaly import (
    calibrate_threshold,
    confusion,
    error_maps,
    healthy_baseline,
    image_auroc,
    image_scores,
    image_scores_pooled,
    pixel_auroc,
    reconstruct,
)
from indusense.vision.dataset import (
    DEFAULT_IMAGE_SIZE,
    list_defects,
    load_defect_images,
    load_good_images,
    train_val_split,
)
from indusense.vision.train import load_trained_model

IMAGE_SIZE = DEFAULT_IMAGE_SIZE
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

model = load_trained_model()
display(Markdown(f"- **Modèle chargé** : {model.count_params()} paramètres."))

- **Modèle chargé** : 57235 paramètres.

## Décision — reconstituer le split de validation et le jeu de test

Même choix qu'aux notebooks 05/06 : recalculer le split (`val_fraction=0.15`,
`seed=42`) et le jeu de test complet plutôt que de dépendre d'un fichier
intermédiaire — chaque notebook reste un livrable autonome.

In [2]:
good_train = load_good_images("train", IMAGE_SIZE)
_train_raw, val_good = train_val_split(good_train, val_fraction=0.15, seed=42)
maps_val = error_maps(val_good, reconstruct(model, val_good))

test_good = load_good_images("test", IMAGE_SIZE)
defects = list_defects()
defect_images = {defect: load_defect_images(defect, IMAGE_SIZE) for defect in defects}
test_images = np.concatenate([test_good, *defect_images.values()], axis=0)
labels = np.concatenate(
    [np.zeros(len(test_good))] + [np.ones(len(images)) for images in defect_images.values()]
)
maps_test = error_maps(test_images, reconstruct(model, test_images))

display(
    Markdown(
        f"- **Validation saine** : {len(val_good)} images.\n"
        f"- **Jeu de test** : {len(test_images)} images "
        f"({len(test_good)} saines, {int(labels.sum())} défauts)."
    )
)

- **Validation saine** : 40 images.
- **Jeu de test** : 167 images (26 saines, 141 défauts).

## Décision — calculer l'ancien score ET le nouveau, en parallèle

**Décision** : garder `image_scores` (ancien, moyenne globale) actif à côté de
`image_scores_pooled` (nouveau), plutôt que de remplacer silencieusement.

**Pourquoi** : le but de ce notebook est de démontrer honnêtement ce que corrige le
nouveau score et ce qu'il ne corrige pas — comparer nécessite de recalculer les deux
sur les mêmes images, pas de se fier à la mémoire des résultats du notebook 05/06.

In [3]:
# Ancien score (notebook 05)
scores_val_old = image_scores(maps_val)
threshold_old = calibrate_threshold(scores_val_old, method="percentile", q=99.0)
scores_test_old = image_scores(maps_test)

# Nouveau score (Phase 1)
baseline = healthy_baseline(maps_val)
scores_val_new = image_scores_pooled(maps_val, baseline, q=99.5)
threshold_new = calibrate_threshold(scores_val_new, method="percentile", q=99.0)
scores_test_new = image_scores_pooled(maps_test, baseline, q=99.5)

display(
    Markdown(
        f"- **Seuil ancien score** (moyenne globale, percentile 99) : {threshold_old:.6f}.\n"
        f"- **Seuil nouveau score** (centile 99,5 après soustraction de la "
        f"référence saine, percentile 99) : {threshold_new:.6f}."
    )
)

- **Seuil ancien score** (moyenne globale, percentile 99) : 0.000291.
- **Seuil nouveau score** (centile 99,5 après soustraction de la référence saine, percentile 99) : 0.001641.

## Livrable — comparaison avant/après par catégorie

Rappel par catégorie de défaut, ancien score vs nouveau score, au seuil calibré de
chacun (percentile 99 sur la validation saine correspondante).

In [4]:
lines = ["- **catégorie** : rappel ancien score → rappel nouveau score"]
offset = len(test_good)
for defect in defects:
    n = len(defect_images[defect])
    recall_old = (scores_test_old[offset : offset + n] > threshold_old).mean()
    recall_new = (scores_test_new[offset : offset + n] > threshold_new).mean()
    lines.append(f"- **{defect}** ({n} images) : {recall_old:.1%} → {recall_new:.1%}")
    offset += n
display(Markdown(chr(10).join(lines)))

- **catégorie** : rappel ancien score → rappel nouveau score
- **color** (25 images) : 40.0% → 72.0%
- **combined** (17 images) : 29.4% → 70.6%
- **contamination** (21 images) : 0.0% → 0.0%
- **crack** (26 images) : 0.0% → 3.8%
- **faulty_imprint** (19 images) : 0.0% → 5.3%
- **pill_type** (9 images) : 100.0% → 100.0%
- **scratch** (24 images) : 0.0% → 0.0%

## Livrable — AUROC et taux de fausses alertes, ancien vs nouveau

In [5]:
auroc_old = image_auroc(scores_test_old, labels)
auroc_new = image_auroc(scores_test_new, labels)

cm_old = confusion(scores_test_old, labels, threshold_old)
cm_new = confusion(scores_test_new, labels, threshold_new)
fp_old = cm_old[0, 1]
fp_new = cm_new[0, 1]

lines = [
    f"- **AUROC image-level** : {auroc_old:.3f} (ancien) → {auroc_new:.3f} (nouveau).",
    f"- **Fausses alertes sur les saines de test** : {fp_old}/{len(test_good)} (ancien) "
    f"→ {fp_new}/{len(test_good)} (nouveau).",
]
display(Markdown(chr(10).join(lines)))

- **AUROC image-level** : 0.688 (ancien) → 0.756 (nouveau).
- **Fausses alertes sur les saines de test** : 0/26 (ancien) → 0/26 (nouveau).

## Conclusion — ce que corrige la Phase 1, et ce qu'elle ne corrige pas

Le nouveau score améliore nettement `color` et `combined` (déjà partiellement
détectées) et l'AUROC image-level global, **sans introduire de nouvelle fausse
alerte** sur les saines de test. C'est un gain réel, obtenu sans réentraînement.

**Mais les 4 catégories cibles (`contamination`, `crack`, `faulty_imprint`,
`scratch`) restent quasiment non détectées.** Conforme au diagnostic du notebook 07 :
leur signal de reconstruction est intrinsèquement trop proche du bruit de fond du
modèle, quelle que soit la façon dont le score est calculé a posteriori — un problème
de score ne peut pas compenser un problème de signal. La Phase 2 (notebook 09,
PatchCore) change d'approche pour s'attaquer directement à cette limite.